In [ ]:
import pandas as pd
import os
import io
import re
from io import BytesIO
from ftplib import FTP
from dotenv import load_dotenv
load_dotenv()

In [27]:
def ConnectFTP(server, username, password):
    """
    Descripcion: Esta funcion establece una conexion FTP con el servidor especificado.
    Parametros:
    server (str): La direccion del servidor FTP.
    username (str): El nombre de usuario para la conexion FTP.
    password (str): La contrasena para la conexion FTP.
    Retorna: Un objeto FTP conectado al servidor.
    """
    ftp = FTP(timeout=60)                
    ftp.connect(server, 21)               
    ftp.login(user=username, passwd=password)

    ftp.set_pasv(True)                    
    ftp.voidcmd("TYPE I")                 

    print(f"Conectado a {server}")
    return ftp

def UploadCsvToFTP(df, path):
    """
    Descripcion: Esta funcion sube un DataFrame de pandas como un archivo CSV a un servidor FTP.
    Parametros:
    df (DataFrame): El DataFrame que se desea subir.
    path (str): La ruta en el servidor FTP donde se guardará el archivo CSV.
    Retorna: None
    """
    csv_buffer = io.BytesIO()
    df.to_csv(csv_buffer, index=False, encoding='utf-8')
    csv_buffer.seek(0)
    ftp = ConnectFTP(os.getenv('ftp_server'),os.getenv('ftp_user'),os.getenv('ftp_password'))
    # remote_path = f"/pruebas/csv/{path}"
    ftp.storbinary(f"STOR {path}", csv_buffer)
    ftp.quit()
    print("Archivo subido correctamente:", path)

def ReadCsvFromFTP(aux, namearc ):
    '''
    Descripcion: Esta funcion lee un archivo CSV desde un servidor FTP y lo carga en un DataFrame de pandas.
    Parametros:
    ftp (FTP): Un objeto FTP conectado al servidor.
    remote_file_path (str): La ruta del archivo CSV en el servidor FTP.
    Retorna: Un DataFrame de pandas que contiene los datos del archivo CSV.
    '''
    ftp = ConnectFTP(os.getenv('ftp_server'), os.getenv('ftp_user'), os.getenv('ftp_password'))
    with BytesIO() as bio:
        ftp.retrbinary(f'RETR /pruebas/csv/{aux}/{namearc}', bio.write)
        bio.seek(0)
        df = pd.read_csv(bio)
    return df

In [28]:
def Mod_sust(df):
    columnas_a_convertir = ["Tabaco", "Alcohol", "Marihuana", "Metanfetaminas", "Otras Sustancias", "Cocaína", "Inhalables", "Alucinógenos", "Medicamentos","Opioides","Nuevas Sustancias Psicoactivas"]
    
    for c in columnas_a_convertir:
        df[c] = df[c].astype(str).str.lower().map({"true": 1, "false": 0, "nan": None})

    for c in columnas_a_convertir:
        df[c] = df[c].astype('Int8') 
    
    return df

def Mod_Sociodemograficos(df):
    list_columns = ['ComunOcupacionId', 'SexoId', 'ComunEstadoCivilId', 'ComunEscolaridadId', 'ComunEstratoSocialId']
    df[list_columns] = df[list_columns].fillna('Sin Dato')

    df['ComunOcupacionId'] = df['ComunOcupacionId'].replace({'Pensionado o jubilado': 1,
                                                    'Sin ocupación': 2, 'Sin ocupacion': 2,
                                                    'Con actividad laboral': 3,
                                                    'Estudiante': 4,
                                                    'Hogar': 5,
                                                    'Sin Dato': 0}).astype('Int8')
    
    df['SexoId'] = df['SexoId'].replace({'Hombre': 1,
                                        'Mujer': 2,
                                        'Sin Dato': 0}).astype('Int8')
    
    df['ComunEstadoCivilId'] = df['ComunEstadoCivilId'].replace({'Soltero(a)': 1,
                                                                'Casado(a)': 2,
                                                                'Divorciado(a)': 3,
                                                                'Unión libre': 4,
                                                                'Unión Libre': 4,
                                                                'Separado(a)': 5,
                                                                'Viudo(a)': 6,
                                                                'Sin Dato': 0}).astype('Int8')
    
    df['ComunEscolaridadId'] = df['ComunEscolaridadId'].replace({'Sin Estudios': 1,
                                                                'Primaria': 2,
                                                                'Secundaria': 3,
                                                                'Preparatoria o Carrera Técnica': 4,
                                                                'Estudios Superiores': 5,
                                                                'Estudios de Posgrado': 6,
                                                                'Sin Dato': 0}).astype('Int8')
    
    df['ComunEstratoSocialId'] = df['ComunEstratoSocialId'].replace({'Muy Bajo': 1,
                                                                    'Bajo': 2,
                                                                    'Medio Bajo': 3,
                                                                    'Medio Alto': 4,                                                                   
                                                                    'Alto': 5,
                                                                    'Sin Dato': 0}).astype('Int8')
    
    df['CentroCostoId'] = df['CentroCostoId'].fillna(99999).astype('Int32')
    return df

def DataClean(df):
    columnas_a_convertir3 = [
        "SexoId",
        "Migracion",
        "ComunEstadoCivilId",
        "ComunEscolaridadId",
        "ComunOcupacionId",
        "ComunEstratoSocialId",
        "PerteneceComunidadLGBTTTI",
        "PerteneceComunidadIndigena",
        "PoblacionAfromexicanaAfroamericana",
        "DiscapacidadPerceptual",
        "Tabaco",
        "Alcohol",
        "Marihuana",
        "Metanfetaminas",
        "Otras Sustancias",
        "ConsumoDeDrogas",
        "ConsumoDeBebidasAlcoholicas",
        "ConsumoDeTabaco",
        "Ludopatia",
        "Otro",
        "TrastornosMentales",
        "Depresion",
        "Psicosis",
        "Epilepsia",
        "Demencia",
        "Autolesion",
        "Suicidio",
        "Ansiedad",
        "ProblemasSalud",
        "ProblemasFamiliares",
        "AccidentesAsociados",
        "ProblemasEscolares",
        "ProblemasLaborales",
        "ProblemasPsicologicos",
        "ProblemasLegales",
        "ConductaAntisocial",
        "ProblemasOtros",
        "Cocaína",
        "Inhalables",
        "Alucinógenos",
        "Medicamentos",
        "Opioides",
        "Nuevas Sustancias Psicoactivas"
    ]
    for columna in columnas_a_convertir3:
        if columna in df.columns:
            df[columna] = pd.to_numeric(df[columna], errors="coerce").astype("Int8")

    df['Año'] = pd.to_numeric(df['Año'], errors="coerce").astype("Int16")
    df['FolioId'] = pd.to_numeric(df['FolioId'], errors="coerce").astype("Int32")

    columnas_eliminar = [
        "ConsumoDeDrogas",
        "ConsumoDeBebidasAlcoholicas",
        "ConsumoDeTabaco",
        "Ludopatia",
        "Otro",
        "TrastornosMentales",
        "Depresion",
        "Psicosis",
        "Epilepsia",
        "Demencia",
        "Autolesion",
        "Suicidio",
        "Ansiedad",
        "Edad_Años",
        'Edad_años',
        "Mes"
        ]
    columnas_eliminar2 = [col for col in df.columns if col.startswith('EdadInicio')] # Posible actualización por agregar estas columnas
    columnas_eliminar.extend(columnas_eliminar2)
    df.drop(columnas_eliminar, axis=1, inplace=True)
    df.drop(df[df['Año'] < 2004].index, inplace=True)
    df = df[df['Estado'].notna() & (df['Estado'] != "")]
    return df

def limpieza_datasets (df):
        #Se definen las columnas que serán filtradas del dataset
        filtro = [
            "Alcohol",
            "Alucinógenos",
            "Cocaína",
            "Inhalables",
            "Marihuana",
            "Medicamentos",
            "Metanfetaminas",
            "Opioides",
            "Otras Sustancias",
            "Tabaco",
            "Nuevas Sustancias Psicoactivas"
        ]
    # Verificar que las columnas existan
        columnas_existentes = [col for col in filtro if col in df.columns]
        # Filtrar filas en el mismo DataFrame (reasignando al índice filtrado)
        df.drop(df[~df[columnas_existentes].ge(1).any(axis=1)].index, inplace=True)

In [29]:
def main():
    list_files = ['GPOS-UM-SLI.csv', 'GPOS-DI-SLI.csv', 'GPOS-AV-SLI.csv', 'GPOS-UM-SI.csv', 'GPOS-DI-SI.csv', 'GPOS-AV-SI.csv']
    for file in list_files:
        print (f'procesando {file}')
        try:
            df = ReadCsvFromFTP('EntrevistaInicialSets', file)
        except Exception as e:
            print(f"8) Opt_GPOS al leer {file}", e)
            return
        try:
            df = Mod_sust(df)
        except Exception as e:
            print(f"8) Opt_GPOS en Mod_sust {file}", e)
            return
        try:
            df = Mod_Sociodemograficos(df)
        except Exception as e:
            print(f"8) Opt_GPOS en Mod_Sociodemograficos {file}", e)
            return
        try:
            df = DataClean(df)
        except Exception as e:
            print(f"8) Opt_GPOS en DataClean {file}", e)
            return
        try:
            limpieza_datasets(df)
        except Exception as e:
            print(f"8) Opt_GPOS en limpieza_datasets {file}", e)
            return
        try:
            UploadCsvToFTP(df, f'/pruebas/csv/carter/{file}')
        except Exception as e:
            print(f"8) Opt_GPOS en CargarFTP {file}", e)
            return
    return df

In [30]:
if __name__ == "__main__":
    df = main()

procesando GPOS-UM-SLI.csv
Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_21664\964868202.py:49: DtypeWarning: Columns (0,17,22,46,47,48,49,50,51,52,53,54,55,62) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)
C:\Users\franc\AppData\Local\Temp\ipykernel_21664\3760355353.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['ComunOcupacionId'] = df['ComunOcupacionId'].replace({'Pensionado o jubilado': 1,
C:\Users\franc\AppData\Local\Temp\ipykernel_21664\3760355353.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', Tr

Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/carter/GPOS-UM-SLI.csv
procesando GPOS-DI-SLI.csv
Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_21664\964868202.py:49: DtypeWarning: Columns (0,17,22,62) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)
C:\Users\franc\AppData\Local\Temp\ipykernel_21664\3760355353.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['ComunOcupacionId'] = df['ComunOcupacionId'].replace({'Pensionado o jubilado': 1,
C:\Users\franc\AppData\Local\Temp\ipykernel_21664\3760355353.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['SexoId'] = df['Sexo

Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/carter/GPOS-DI-SLI.csv
procesando GPOS-AV-SLI.csv
Conectado a 192.168.62.13


...

Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/carter/GPOS-UM-SI.csv
procesando GPOS-DI-SI.csv
Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_21664\964868202.py:49: DtypeWarning: Columns (0,17,22,62) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)
C:\Users\franc\AppData\Local\Temp\ipykernel_21664\3760355353.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['ComunOcupacionId'] = df['ComunOcupacionId'].replace({'Pensionado o jubilado': 1,
C:\Users\franc\AppData\Local\Temp\ipykernel_21664\3760355353.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['SexoId'] = df['Sexo

Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/carter/GPOS-DI-SI.csv
procesando GPOS-AV-SI.csv
Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_21664\964868202.py:49: DtypeWarning: Columns (0,12,14,16,19,21,22,27,51,52,53,54,55,62) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)
C:\Users\franc\AppData\Local\Temp\ipykernel_21664\3760355353.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['ComunOcupacionId'] = df['ComunOcupacionId'].replace({'Pensionado o jubilado': 1,
C:\Users\franc\AppData\Local\Temp\ipykernel_21664\3760355353.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', Tr

Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/carter/GPOS-AV-SI.csv
